# FPL API Ingestion
This notebook focuses on fetching data from the Fantasy Premier League (FPL) API and storing it in Parquet format for efficient processing in subsequent notebooks.

## 1. Install Required Libraries
We need the `requests` library to interact with the FPL API, `pandas` for data manipulation, and `pyarrow` for Parquet file handling.

In [ ]:
import requests
import pandas as pd
import os
import time
from datetime import datetime

## 2. API Interaction
The `fetch_fpl_data` function sends GET requests to the FPL API endpoints and returns the JSON response.

In [ ]:
def fetch_fpl_data(endpoint, max_retries=3, base_wait_time=1):
    """Fetches data from the FPL API with exponential backoff.

    Args:
        endpoint (str): The API endpoint to fetch.
        max_retries (int): The maximum number of times to retry the request.
        base_wait_time (int): The initial wait time in seconds for the first retry.

    Returns:
        dict: The JSON response from the API.
        
    Raises:
        requests.exceptions.RequestException: If the request fails after all retries.
    """

    base_url = "https://fantasy.premierleague.com/api/"
    url = base_url + endpoint

    # Initialize the current wait time
    wait_time = base_wait_time

    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10) # Added a timeout for safety
            
            # Raise an exception for bad status codes (e.g., 429 Too Many Requests, 5xx server errors)
            response.raise_for_status() 
            
            return response.json()  # If successful, return the data and exit the function

        except requests.exceptions.RequestException as e:
            print(f"An error occurred: {e}")

            # If this was the last attempt, print a final message and re-raise the exception
            if attempt == max_retries - 1:
                print("Max retries reached. Giving up.")
                raise

            sleep_duration = wait_time
            
            print(f"Waiting for {sleep_duration:.2f} seconds before retrying...")
            time.sleep(sleep_duration)
            
            # Double the wait time for the next attempt
            wait_time *= 2

## 3. Fetch Data from Various Endpoints
We fetch data from the `bootstrap-static`, `element-summary`, and `fixtures` endpoints.

In [4]:
# Fetch data from various endpoints
bootstrap_data = fetch_fpl_data("bootstrap-static/")
elements_data = bootstrap_data['elements']
teams_data = bootstrap_data['teams']
events_data = bootstrap_data['events']

# Convert to Pandas DataFrames
elements_df = pd.DataFrame(elements_data)
teams_df = pd.DataFrame(teams_data)
events_df = pd.DataFrame(events_data)

# Fetch player summary data
player_summary_data = {}
for player_id in elements_df['id']:
    player_summary_data[player_id] = fetch_fpl_data(f"element-summary/{player_id}/")

# Extract history from player summary data
player_history_data = {}
for player_id, data in player_summary_data.items():
    player_history_data[player_id] = data['history']

# Convert player history to a DataFrame
player_history_dfs = []
for player_id, history in player_history_data.items():
    temp_df = pd.DataFrame(history)
    temp_df['element'] = player_id  # Add player ID as a column
    player_history_dfs.append(temp_df)

player_history_df = pd.concat(player_history_dfs, ignore_index=True)

# Fetch fixture data
fixture_data = {}
for team_id in teams_df['id']:
    fixture_data[team_id] = fetch_fpl_data(f"fixtures/?team={team_id}")

# Extract fixture data from team fixture data
fixtures_dfs = []
for team_id, fixtures in fixture_data.items():
    temp_df = pd.DataFrame(fixtures)
    temp_df['team_id'] = team_id
    fixtures_dfs.append(temp_df)

fixtures_df = pd.concat(fixtures_dfs, ignore_index=True)

## 4. Data Structuring and Storage
We create a directory to store the data and save each DataFrame as a Parquet file.

In [5]:
# Create a directory for storing data (if it doesn't exist)
data_source = "fpl_api"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
data_dir = f"../data/raw/{data_source}/{current_datetime}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Save DataFrames as CSV files
elements_df.to_csv(os.path.join(data_dir, "2025_26_elements.csv"), index=False)
teams_df.to_csv(os.path.join(data_dir, "2025_26_teams.csv"), index=False)
events_df.to_csv(os.path.join(data_dir, "2025_26_events.csv"), index=False)
player_history_df.to_csv(os.path.join(data_dir, "2025_26_player_history.csv"), index=False)
fixtures_df.to_csv(os.path.join(data_dir, "2025_26_fixtures.csv"), index=False)

print(f"Data ingestion complete. Data saved to {data_dir}.")

Data ingestion complete. Data saved to ../data/raw/fpl_api/20250821_211214.
